In [1]:
# Required Imports
import os
import sys
import numpy as np
import tensorflow as tf
from keras.models import load_model
from keras.preprocessing.text import tokenizer_from_json
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import json
from tqdm import tqdm
import logging

In [2]:
# Suppress TensorFlow warnings and logs
tf.get_logger().setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

print("Starting BLEU Score Analysis Script...")

Starting BLEU Score Analysis Script...


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive Mounted Successfully.")

sys.path.insert(0, '/content/drive/MyDrive/CSCK507_EMA_GroupC_Final')
import config

model_file = os.path.join(config.MAIN_DIR_PATH, config.SEQ2SEQ_NA_MODEL)
tokenizer_file = os.path.join(config.MAIN_DIR_PATH, config.TOKENIZER_FILE)
test_data_file = os.path.join(config.MAIN_DIR_PATH, config.TEST_DATA)

print(f"Model File Path: {model_file}")
print(f"Tokenizer File Path: {tokenizer_file}")
print(f"Test Data File Path: {test_data_file}")

Mounted at /content/drive
Google Drive Mounted Successfully.
Model File Path: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/seq2seq_no_attention_model.h5
Tokenizer File Path: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/tokenizer.json
Test Data File Path: /content/drive/MyDrive/CSCK507_EMA_GroupC_Final/test_data.npz


In [4]:
# Load the Pre-trained Model
seq2seq_model = load_model(model_file)
print("Model loaded successfully!")

# Print the model summary to verify its structure
seq2seq_model.summary()

Model loaded successfully!
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 Encoder-Input (InputLayer)  [(None, 30)]                 0         []                            
                                                                                                  
 Decoder-Input (InputLayer)  [(None, 30)]                 0         []                            
                                                                                                  
 Encoder-Embedding (Embeddi  (None, 30, 64)               4099366   ['Encoder-Input[0][0]']       
 ng)                                                      4                                       
                                                                                                  
 Decoder-Embedding (Embeddi  (None, 30, 64)               4099366  

In [5]:
# Load the tokenizer
with open(tokenizer_file, 'r') as f:
    tokenizer_data = json.load(f)
    tokenizer = tf.keras.preprocessing.text.tokenizer_from_json(tokenizer_data)

# Create index mappings using the correct word-level index
target_token_index = tokenizer.word_index
reverse_target_word_index = {i: word for word, i in target_token_index.items()}  # Ensure word-level mapping

print(f"Tokenizer Loaded. Number of Tokens: {len(target_token_index)}")

Tokenizer Loaded. Number of Tokens: 640525


In [6]:
# Load the Test Data
test_data = np.load(test_data_file)
X_test_enc, X_test_dec, y_test = test_data['X_test_enc'], test_data['X_test_dec'], test_data['y_test']
print(f"Test Data Loaded. Number of Test Samples: {X_test_enc.shape[0]}")

Test Data Loaded. Number of Test Samples: 2025386


In [7]:
# Extract Encoder and Decoder Models from the Main Model

# Extract Encoder Inputs and States
encoder_inputs = seq2seq_model.input[0]
encoder_lstm_layer = seq2seq_model.get_layer("Encoder-LSTM")
encoder_outputs, state_h_enc, state_c_enc = encoder_lstm_layer.output

# Create the Encoder Model using inputs and state outputs
encoder_model = tf.keras.Model(inputs=encoder_inputs, outputs=[state_h_enc, state_c_enc])
print("Encoder Model Created Successfully.")

# Extract Decoder Inputs and States
decoder_inputs = seq2seq_model.input[1]

# Add Decoder Embedding Layer
decoder_embedding_layer = seq2seq_model.get_layer("Decoder-Embedding")
decoder_embedding_outputs = decoder_embedding_layer(decoder_inputs)

# Prepare Initial States for the Decoder LSTM with bfloat16 precision
decoder_state_input_h = tf.keras.Input(shape=(64,), name='input_3', dtype='bfloat16')  # Hidden state input for decoder
decoder_state_input_c = tf.keras.Input(shape=(64,), name='input_4', dtype='bfloat16')  # Cell state input for decoder

# Reuse the existing Decoder LSTM and Dense layers
decoder_lstm_layer = seq2seq_model.get_layer("Decoder-LSTM")
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm_layer(
    decoder_embedding_outputs, initial_state=[decoder_state_input_h, decoder_state_input_c])

decoder_dense_layer = seq2seq_model.get_layer("Decoder-Output")
decoder_outputs = decoder_dense_layer(decoder_outputs)

# Create the Decoder Model using inputs and state outputs
decoder_model = tf.keras.Model([decoder_inputs] + [decoder_state_input_h, decoder_state_input_c], [decoder_outputs, state_h_dec, state_c_dec])
print("Decoder Model Created Successfully.")

Encoder Model Created Successfully.
Decoder Model Created Successfully.


In [13]:
# BLEU Score Analysis Function in batches

def batch_greedy_decode(input_seqs, encoder_model, decoder_model, target_token_index, reverse_target_word_index, max_decoder_seq_length):
    """
    Perform Greedy Search to decode a batch of sequences using the encoder and decoder models.
    """
    batch_size = input_seqs.shape[0]
    states_values = encoder_model.predict(input_seqs, verbose=0)

    start_token_index = target_token_index.get('<SOS>', 0)
    target_seqs = np.full((batch_size, 1), start_token_index)

    decoded_sentences = [[] for _ in range(batch_size)]
    finished = [False] * batch_size

    for _ in range(max_decoder_seq_length):
        output_tokens, h, c = decoder_model.predict([target_seqs] + states_values, verbose=0)
        sampled_token_indices = np.argmax(output_tokens[:, -1, :], axis=1)

        for i, sampled_token_index in enumerate(sampled_token_indices):
            if not finished[i]:
                sampled_word = reverse_target_word_index.get(sampled_token_index, '')

                if sampled_word == '<EOS>':
                    finished[i] = True
                else:
                    decoded_sentences[i].append(sampled_word)

        target_seqs = sampled_token_indices[:, np.newaxis]
        states_values = [h, c]

    return [' '.join(sentence) for sentence in decoded_sentences]

def decode_indices_to_words(sequence, reverse_target_word_index):
    """
    Convert a sequence of indices to a readable string using the reverse target word index.

    Parameters:
    - sequence: List of word indices.
    - reverse_target_word_index: Dictionary mapping indices to words.

    Returns:
    - Decoded sentence.
    """
    words = []
    for index in sequence:
        word = reverse_target_word_index.get(index, '')
        if word == '<EOS>':
            break
        words.append(word)
    return ' '.join(words).strip()

In [17]:
# Evaluate the Model Using BLEU Score

subset_size=2000
batch_size=10
bleu_scores = []

running_sum = 0

for i in tqdm(range(0, subset_size, batch_size), desc="Calculating BLEU Scores"):
    batch_end = min(i + batch_size, subset_size)
    input_seqs = X_test_enc[i:batch_end]

    # Generate the candidate sequences for the batch
    candidate_sentences = batch_greedy_decode(input_seqs, encoder_model, decoder_model, target_token_index, reverse_target_word_index, max_decoder_seq_length=30)

    # Calculate BLEU scores for each sequence in the batch
    for j, candidate_sentence in enumerate(candidate_sentences):
        reference_sentence = decode_indices_to_words(y_test[i + j], reverse_target_word_index)
        reference = [reference_sentence.split()]
        candidate = candidate_sentence.split()

        bleu_score = sentence_bleu(reference, candidate, smoothing_function=SmoothingFunction().method1)
        bleu_scores.append(bleu_score)
        running_sum += bleu_score

    # Print real-time average BLEU score at a reduced frequency
    if (i // batch_size) % 5 == 0:  # Print after every 5 batches
        print(f"Real-time Average BLEU Score after {i + batch_size} samples: {running_sum / len(bleu_scores):.4f}")

average_bleu_score = np.mean(bleu_scores)
print(f"\nFinal Average BLEU Score for the Subset of Test Data: {average_bleu_score:.4f}")

Calculating BLEU Scores:   0%|          | 1/200 [00:06<22:42,  6.85s/it]

Real-time Average BLEU Score after 10 samples: 0.0179


Calculating BLEU Scores:   3%|▎         | 6/200 [00:40<21:54,  6.78s/it]

Real-time Average BLEU Score after 60 samples: 0.0091


Calculating BLEU Scores:   6%|▌         | 11/200 [01:14<21:23,  6.79s/it]

...

Real-time Average BLEU Score after 1910 samples: 0.0121


Calculating BLEU Scores:  98%|█████████▊| 196/200 [22:28<00:27,  6.88s/it]

Real-time Average BLEU Score after 1960 samples: 0.0121


Calculating BLEU Scores: 100%|██████████| 200/200 [22:56<00:00,  6.88s/it]


Final Average BLEU Score for the Subset of Test Data: 0.0121
